In [ ]:
import os, sys
import torch, json
import numpy as np

os.chdir("/home/sonat/OCR_HDV/") # TODO: Update to your project path.

from util.slconfig import SLConfig

from datasets import build_dataset
from util.visualizer import COCOVisualizer
from util import box_ops
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
model_config_path = "config/finetuning.py" # TODO: Update to your model config path.
model_checkpoint_path = "logs/finetune.pth" # TODO: Update to your checkpoint path.

In [ ]:
from pretraining import build_model_main as build_model
args = SLConfig.fromfile(model_config_path) 
args.device = 'cuda' 
model, criterion, postprocessors = build_model(args)
checkpoint = torch.load(model_checkpoint_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])
print("Epoch {}: loaded model weights from {}".format(checkpoint['epoch'], model_checkpoint_path))
_ = model.eval()

args.dataset_file = 'eida'
args.coco_path = "/comp_robot/cv_public_dataset/COCO2017/" # the path of coco
args.fix_size = True

image_set = "test" # TODO: Update to your desired image set (e.g., "train", "val", "test").
args.query_dim=64
dataset_val = build_dataset(image_set, args=args)   

In [ ]:
import numpy as np
import torch
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def compute_iou(pred_poly, gt_poly):
    """Simple IoU computation between two polygons"""
    # Convert to numpy and reshape to (N,2) format

    if torch.is_tensor(pred_poly):
        pred_poly = pred_poly.detach().cpu().numpy()
    if torch.is_tensor(gt_poly):
        gt_poly = gt_poly.detach().cpu().numpy()
    
    # Reshape to (N,2) format
    pred_poly = pred_poly.reshape(-1, 2)
    gt_poly = gt_poly.reshape(-1, 2)
        # Create Shapely polygons
    pred_shapely = Polygon(pred_poly)
    gt_shapely = Polygon(gt_poly)

    # Handle invalid polygons
    if not pred_shapely.is_valid:
        pred_shapely = pred_shapely.buffer(0)
    if not gt_shapely.is_valid:
        gt_shapely = gt_shapely.buffer(0)
        

    intersection = pred_shapely.intersection(gt_shapely).area
    union = pred_shapely.union(gt_shapely).area
    iou = intersection / union if union > 0 else 0

        
    return iou

def match_predictions_with_gts(pred_polygons, pred_scores, gt_polygons, iou_threshold=0.5):
    """Match predictions to ground truths using IoU"""
    # Convert inputs to numpy if needed
    if torch.is_tensor(pred_scores):
        pred_scores = pred_scores.detach().cpu().numpy()
    
    num_preds = len(pred_polygons)
    num_gts = len(gt_polygons)
    
    # Compute IoU matrix
    iou_matrix = np.zeros((num_preds, num_gts))
    for i in range(num_preds):
        for j in range(num_gts):
            iou_matrix[i, j] = compute_iou(pred_polygons[i], gt_polygons[j])

    # Initialize lists for matching
    matched = []
    unmatched_preds = list(range(num_preds))
    unmatched_gts = list(range(num_gts))
    
    # Sort predictions by confidence
    pred_indices = np.argsort(-pred_scores)
    
    # Match predictions to ground truths

    for pred_idx in pred_indices:

        if pred_idx in unmatched_preds:

            # Get IoUs for this prediction
            ious = iou_matrix[pred_idx]
                
            # Find best matching ground truth
            best_gt_idx = np.argmax(ious)
            best_iou = ious[best_gt_idx]
  
            # If IoU is above threshold and GT is unmatched, make the match
            if best_iou >= iou_threshold and best_gt_idx in unmatched_gts:
                matched.append((pred_idx, best_gt_idx))
                unmatched_preds.remove(pred_idx)
                unmatched_gts.remove(best_gt_idx)

    return matched, unmatched_preds, unmatched_gts

def renorm(img: torch.FloatTensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) \
        -> torch.FloatTensor:
    # img: tensor(3,H,W) or tensor(B,3,H,W)
    # return: same as img
    assert img.dim() == 3 or img.dim() == 4, "img.dim() should be 3 or 4 but %d" % img.dim() 
    if img.dim() == 3:
  
        assert img.size(0) == 3, 'img.size(0) shoule be 3 but "%d". (%s)' % (img.size(0), str(img.size()))
        img_perm = img.permute(1,2,0)
        mean = torch.Tensor(mean)
        std = torch.Tensor(std)
        img_res = img_perm * std + mean
        return img_res.permute(2,0,1)
    else: # img.dim() == 4
        assert img.size(1) == 3, 'img.size(1) shoule be 3 but "%d". (%s)' % (img.size(1), str(img.size()))
        img_perm = img.permute(0,2,3,1)
        mean = torch.Tensor(mean)
        std = torch.Tensor(std)
        img_res = img_perm * std + mean
        return img_res.permute(0,3,1,2)


def is_reading_order_correct(pred_poly, gt_poly, key_indices=[0, 15, 16, 31]):
    """
    Returns 1 if *all* four key corners match in reading order; 0 otherwise.
    
    We take inspiration from the 'compute_accuracy_order' logic. 
    The 'key_indices' map the predicted corners. 
    The 'key_gt' map tells which GT corner index each predicted corner 
    is supposed to match.

    If all four corners match, we say the reading order is correct (return 1). 
    Otherwise, return 0.
    """
    pred_coords = pred_poly.cpu().numpy().reshape(-1, 2)
    gt_coords   = gt_poly.cpu().numpy().reshape(-1, 2)


    K = gt_coords.shape[0]//2-1
    key_gt_indices = [0, K, K+1, 2*K+1]

    key_gt = {key_indices[0]: [key_gt_indices[0]], 
              key_indices[1]: [key_gt_indices[1]],
              key_indices[2]: [key_gt_indices[2]], 
              key_indices[3]: [key_gt_indices[3]]}  

    local_score = 0.0
    for key_i in key_indices:
        pred_point = pred_coords[key_i]  # [x, y]
        
        # Find nearest corner in GT
        min_dist = float('inf')
        min_dist_idx = -1
        for j, gt_point in enumerate(gt_coords):
            if j not in key_gt_indices:
                continue
            dist = math.dist(pred_point, gt_point)
            if dist < min_dist:
                min_dist = dist
                min_dist_idx = j
        # If the GT corner index is the "expected" one, increment
        if min_dist_idx in key_gt[key_i]:
            local_score += 1.0

    # local_score now is in [0..4]; normalize by 4 to get fraction
    final_score = local_score / 4.0

    # We say the reading order is "correct" only if the score is 1.0 (i.e., 4/4)
    return 1 if (abs(final_score - 1.0) < 1e-9) else 0

def compute_reading_order_prf(
    matched_pairs,
    unmatched_preds,
    unmatched_gts,
    pred_polygons,
    gt_polygons
):
    # Count how many matched pairs have correct reading order
    ro_tp = 0

    K_ = pred_polygons.shape[1]//4-1
    key_indices = [0, K_, K_+1, 2*K_+1]
    for (pred_idx, gt_idx) in matched_pairs:
        if is_reading_order_correct(pred_polygons[pred_idx], gt_polygons[gt_idx], key_indices=key_indices) == 1:
            ro_tp += 1

    # total matched pairs
    matched_count = len(matched_pairs)

    # A matched pair is "bad" if reading order is not correct
    ro_bad = matched_count - ro_tp  # matched but reading order = 0

    # ro_fp = unmatched predictions + matched pairs that have incorrect order
    ro_fp = len(unmatched_preds) + ro_bad

    # ro_fn = unmatched gts + matched pairs that fail reading order from the GT perspective
    #        (we treat the same ro_bad pairs as failing for the GT).
    ro_fn = len(unmatched_gts) + ro_bad

    # Compute precision, recall
    denom_p = ro_tp + ro_fp
    denom_r = ro_tp + ro_fn
    ro_precision = ro_tp / denom_p if denom_p > 0 else 0.0
    ro_recall    = ro_tp / denom_r if denom_r > 0 else 0.0

    # F1
    denom_f1 = ro_precision + ro_recall
    ro_f1 = 2.0 * ro_precision * ro_recall / denom_f1 if denom_f1 > 0 else 0.0
    return ro_precision, ro_recall, ro_f1, ro_tp, ro_fp, ro_fn


from matplotlib import patheffects
# Assuming your bounding boxes are in the format [xmin, ymin, xmax, ymax]
def display_image_with_boxes(image, polygons, gt_polygons, name, matched):

    matched_preds_list = [pred_idx for (pred_idx, gt_idx) in matched]
    matched_gt_list = [gt_idx for (pred_idx, gt_idx) in matched]
    # Create figure and axes
    fig, ax = plt.subplots(1, figsize=(6,6), dpi=50)

    # Display the image
    ax.imshow(image)

    # Add polygons to the image
    for p, polygon in enumerate(polygons):
        if p not in matched_preds_list:
            box_color = "red"
        else:
            box_color = "darkgreen"
        # Convert list of polygon points to numpy array for easier manipulation
        points = np.array(polygon).reshape(-1, 2)

        # Create a polygon patch
        poly_patch = patches.Polygon(points, linewidth=2., edgecolor=box_color, facecolor='none')
        # Add the patch to the Axes
        ax.add_patch(poly_patch)


        # Baseline points (first half)
        baseline_pts = points[:K]

        # Get midpoint of baseline
        start = baseline_pts[0]
        end = baseline_pts[-1]
        mid = ((start[0] + end[0]) / 2, (start[1] + end[1]) / 2)

        # Compute orientation angle
        dx, dy = end[0] - start[0], end[1] - start[1]
        angle = np.degrees(np.arctan2(dy, dx))

        # Place arrow (use a text symbol like "→")
        arrow_text = "➔"
        txt = ax.text(
            mid[0], mid[1], arrow_text,
            fontsize=12, color=box_color,
            rotation=angle,
            rotation_mode="default",
            ha="center", va="center",
            transform_rotates_text=True  # crucial: rotate in data coords
        )
        txt.set_path_effects([patheffects.withStroke(linewidth=3, foreground=box_color)])

    # Add polygons to the image
    for p, polygon in enumerate(gt_polygons):
        if p not in matched_gt_list:
            box_color = "blue"
            # Convert list of polygon points to numpy array for easier manipulation
            points = np.array(polygon).reshape(-1, 2)
            # Create a polygon patch
            poly_patch = patches.Polygon(points, linewidth=2., edgecolor=box_color, facecolor='none')
            # Add the patch to the Axes
            ax.add_patch(poly_patch)


            # Baseline points (first half)
            baseline_pts = points[:K]

            # Get midpoint of baseline
            start = baseline_pts[0]
            end = baseline_pts[-1]
            mid = ((start[0] + end[0]) / 2, (start[1] + end[1]) / 2)

            # Compute orientation angle
            dx, dy = end[0] - start[0], end[1] - start[1]
            angle = np.degrees(np.arctan2(dy, dx))

            # Place arrow (use a text symbol like "→")
            arrow_text = "➔"
            txt = ax.text(
                mid[0], mid[1], arrow_text,
                fontsize=18, color=box_color,
                rotation=angle,
                rotation_mode="default",
                ha="center", va="center",
                transform_rotates_text=True  # crucial: rotate in data coords
            )
            txt.set_path_effects([patheffects.withStroke(linewidth=3, foreground=box_color)])

    # Show plot
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()


In [ ]:
# Code to evaluate the model on the test set
import torchvision
import math

gt_instances_list = []
pred_instances_list = []

score_th = 0.
nms_th = 0.5

import json
# Load script map
with open("notebook/script_evaluation_map.json") as f:
    script_map = json.load(f)

script = "all" # "chinese" TODO: Update to your desired script.

calculated_img_idx = []
for idx in range(len(dataset_val)):
    image, targets = dataset_val[idx]
    file_name = targets["name"]
    if script == "all":
        calculated_img_idx.append(idx)
    else:
        if not any(f.endswith(file_name) for f in script_map[script]):
            continue
        else:
            calculated_img_idx.append(idx)
    with torch.no_grad():
        model.eval()
        output = model.cuda()(image[None].cuda())
        polygones = output['pred_boxes']
        mask = output['pred_logits'].sigmoid().max(-1)[0] > score_th

    # Scale predicted polygons back to the original image size
    h, w = image.shape[1], image.shape[2]
    scale_tensor = torch.tensor([w, h], dtype=torch.float32)
    scale_tensor = scale_tensor.repeat(polygones.shape[-1] // 2)

    interm_poly = (polygones[mask].cpu().detach() * scale_tensor)
    scores = output['pred_logits'].sigmoid().max(-1)[0][mask]

    # Optional NMS
    x_coords = interm_poly[:, 0::2]
    y_coords = interm_poly[:, 1::2]
    x_min = x_coords.min(-1)[0]
    y_min = y_coords.min(-1)[0]
    x_max = x_coords.max(-1)[0]
    y_max = y_coords.max(-1)[0]

    boxes = torch.stack([x_min, y_min, x_max, y_max], dim=1)

    if nms_th is not None:
        nms_idx = torchvision.ops.nms(boxes.cuda(), scores.cuda(), nms_th).cpu()
        interm_poly = interm_poly[nms_idx]
        scores = scores[nms_idx]

    # Renormalize image for visualization (if needed)
    img_renorm = renorm(image)  # This is your custom function that "renorms"
    img_renorm = img_renorm.permute(1, 2, 0)  # Now it's H x W x C

    # Prepare predicted and ground-truth polygons
    pred_instances = {
        'polygons': interm_poly.cpu().detach(),
        'scores': scores.cpu().numpy(),
    }
    gt_polygons = targets['boxes'] * torch.tensor(
        [w, h]*32,
        dtype=torch.float32
    )
    gt_instances = {
        'polygons': gt_polygons,
        'ignored': []
    }

    gt_instances_list.append(gt_instances)
    pred_instances_list.append(pred_instances)

In [ ]:
iou_threshold = 0.5
tps, fps, fns = 0, 0, 0
ro_tps, ro_fps, ro_fns = 0, 0, 0
all_matches = []
all_scores = []
all_ro_matches = []

counter = 0
for gt_instances, pred_instances in zip(gt_instances_list, pred_instances_list):
    image, targets = dataset_val[calculated_img_idx[counter]]
    counter+=1
    img_renorm = renorm(image)  # This is your custom function that "renorms"
    img_renorm = img_renorm.permute(1, 2, 0)  # Now it's H x W x C

    # Match predictions with ground truths
    matched, unmatched_preds, unmatched_gts = match_predictions_with_gts(
        pred_polygons=pred_instances['polygons'],
        pred_scores=pred_instances['scores'],
        gt_polygons=gt_instances['polygons'],
        iou_threshold=iou_threshold
    )
    # Compute basic metrics
    TP = len(matched)
    FP = len(unmatched_preds)
    FN = len(unmatched_gts)


    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    tps += TP
    fps += FP
    fns += FN

    ro_p, ro_r, ro_f1, ro_tp, ro_fp, ro_fn = compute_reading_order_prf(matched, unmatched_preds, unmatched_gts, pred_instances['polygons'], gt_instances['polygons'])

    ro_tps += ro_tp
    ro_fps += ro_fp
    ro_fns += ro_fn

    interm_poly = pred_instances['polygons']
    gt_polygons = gt_instances['polygons']
    
    # display_image_with_boxes(img_renorm, interm_poly.cpu().detach(), gt_polygons.cpu().detach(), targets["name"], matched)

    K_ = pred_instances['polygons'].shape[1]//4-1
    key_indices = [0, K_, K_+1, 2*K_+1]
    scores = pred_instances['scores']
    for pred_idx, gt_idx in matched:
        all_scores.append(scores[pred_idx].item())
        all_matches.append(1)
        all_ro_matches.append(is_reading_order_correct(
            pred_instances['polygons'][pred_idx],
            gt_instances['polygons'][gt_idx],
            key_indices=key_indices
        ))

    for pred_idx in unmatched_preds:
        all_scores.append(scores[pred_idx].item())
        all_matches.append(0)
        all_ro_matches.append(0)

total_gt = tps + fns

all_scores = np.array(all_scores)
all_matches = np.array(all_matches)
sorted_idx = np.argsort(-all_scores)
all_matches = all_matches[sorted_idx]
all_ro_matches = np.array(all_ro_matches)[sorted_idx]

# Compute precision & recall
tp_cumsum = np.cumsum(all_matches)
fp_cumsum = np.cumsum(1 - all_matches)
precision = tp_cumsum / (tp_cumsum + fp_cumsum)
recall = tp_cumsum / total_gt  # Total GT = sum(all_matches)

tp_cumsum_ro = np.cumsum(all_ro_matches)
fp_cumsum_ro = np.cumsum(1 - all_ro_matches)
precision_ro = tp_cumsum_ro / (tp_cumsum_ro + fp_cumsum_ro)
recall_ro = tp_cumsum_ro / total_gt  # Total GT = sum(all_matches)

# A mask to avoid division by zero if precision + recall is 0 (though unlikely here)
denominator = precision + recall
f1_scores = np.zeros_like(precision) # Initialize an array for F1 scores

denominator_ro = precision_ro + recall_ro
f1_scores_ro = np.zeros_like(precision_ro) # Initialize an array for F1 scores

# Calculate F1 score for all points where the denominator is non-zero
valid_indices = denominator > 0
f1_scores[valid_indices] = 2 * (precision[valid_indices] * recall[valid_indices]) / denominator[valid_indices]

valid_indices_ro = denominator_ro > 0
f1_scores_ro[valid_indices_ro] = 2 * (precision_ro[valid_indices_ro] * recall_ro[valid_indices_ro]) / denominator_ro[valid_indices_ro]

# Find the index of the highest F1 score
max_f1_index = np.argmax(f1_scores)
max_f1_index_ro = np.argmax(f1_scores_ro)

if image_set == "test":
    # TODO: Set threshold obtained from validation set.
    scores_at_max_f1 = 0.3570
    scores_at_max_f1_ro = 0.4699

    max_f1 = f1_scores[np.where(all_scores[sorted_idx] >= scores_at_max_f1)[0][-1]]
    max_f1_precision = precision[np.where(all_scores[sorted_idx] >= scores_at_max_f1)[0][-1]]
    max_f1_recall = recall[np.where(all_scores[sorted_idx] >= scores_at_max_f1)[0][-1]]
    max_f1_ro = f1_scores_ro[np.where(all_scores[sorted_idx] >= scores_at_max_f1_ro)[0][-1]]

    print(f"Max F1 Score: {max_f1:.4f} at Precision: {max_f1_precision:.4f}, Recall: {max_f1_recall:.4f}")
    print(f"Max F1-RO Score: {max_f1_ro:.4f} at Precision: {precision_ro[max_f1_index_ro]:.4f}, Recall: {recall_ro[max_f1_index_ro]:.4f}")


else:
    # Retrieve the maximum F1-score and its corresponding Precision and Recall
    max_f1 = f1_scores[max_f1_index]
    max_f1_precision = precision[max_f1_index]
    max_f1_recall = recall[max_f1_index]
    max_f1_ro = f1_scores_ro[max_f1_index_ro]

    # Retrieve the score threshold at max F1
    max_f1_threshold = all_scores[sorted_idx][max_f1_index]
    max_f1_ro_threshold = all_scores[sorted_idx][max_f1_index_ro]
    print(f"Max F1 Score: {max_f1:.4f} at Precision: {max_f1_precision:.4f}, Recall: {max_f1_recall:.4f}, Threshold: {max_f1_threshold:.4f}")
    print(f"Max F1-RO Score: {max_f1_ro:.4f} at Precision: {precision_ro[max_f1_index_ro]:.4f}, Recall: {recall_ro[max_f1_index_ro]:.4f}, Threshold: {max_f1_ro_threshold:.4f}")


import numpy as np

def calculate_interpolated_ap(recall, precision):
    sort_indices = np.argsort(recall)
    recall = recall[sort_indices]
    precision = precision[sort_indices]

    for i in range(len(precision) - 2, -1, -1):
        precision[i] = np.maximum(precision[i], precision[i + 1])
        
   
    ap = np.trapz(precision, recall)
    
    return ap

mAP50 = calculate_interpolated_ap(recall, precision)
map_ro_50 = calculate_interpolated_ap(recall_ro, precision_ro)
print("AP:", mAP50)
print("AP-RO:", map_ro_50)